# Day 4: Modern ML with HuggingFace Transformers

<a target="_blank" href="https://colab.research.google.com/github/LuWidme/uk259/blob/main/demos/07_HuggingFace_Transformers.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

**Duration:** 1.5-2 hours  
**Prerequisites:** Basic Python, understanding of classification, neural networks intro  

**Learning Objectives:**
- Understand what pre-trained models and transfer learning are
- Use HuggingFace pipelines for common NLP tasks
- Load and explore datasets from the HuggingFace Hub
- Apply sentiment analysis, text generation, and zero-shot classification
- Understand when and why to use pre-trained models

**Libraries Used:** `transformers`, `datasets` (HuggingFace)

---

## Why HuggingFace?

### The Problem: Training From Scratch is Expensive

In the previous notebook we trained a neural network on MNIST. That worked because:
- The task was simple (28x28 pixel images)
- The dataset was small (60,000 images)
- Training took seconds on a laptop

But for real-world tasks like understanding text or recognizing objects in photos:
- Models have **billions** of parameters
- Training requires **thousands of GPUs** for **weeks**
- Training data costs **millions of dollars** to collect

**Example:** Training GPT-3 cost an estimated $4.6 million in compute alone!

### The Solution: Pre-Trained Models

**Transfer learning:** Someone else trains a huge model, and you reuse it for your task.

**Analogy:** You don't need to learn English from scratch to write an essay. You already know the language — you just apply it to a new topic. Pre-trained models work the same way!

### What is HuggingFace?

**HuggingFace** is the "GitHub for machine learning models." It provides:

- **Model Hub**: 500,000+ pre-trained models you can use for free
- **Datasets Hub**: 100,000+ datasets ready to download
- **Transformers Library**: Easy-to-use Python API for all these models
- **Pipelines**: One-line solutions for common tasks

**In this notebook, you'll use models that were trained on massive datasets — with just a few lines of code!**

---

## Setup

The `transformers` and `datasets` libraries should already be installed in your environment.

If not, run: `pip install transformers datasets`

In [ ]:
from transformers import pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("✓ Libraries imported successfully!")
print("\nNote: The first time you use a model, it will be downloaded.")
print("This might take a minute depending on your internet connection.")

---

## Part 1: Sentiment Analysis

**Sentiment analysis** determines whether text expresses a positive or negative feeling.

**Use cases:**
- Analyzing customer reviews
- Monitoring social media sentiment
- Understanding feedback at scale

### Using a HuggingFace Pipeline

A **pipeline** bundles everything you need into one line:
- Downloads the pre-trained model
- Processes your text (tokenization)
- Runs the model
- Returns human-readable results

In [ ]:
# Create a sentiment analysis pipeline
# (First run downloads the model — ~260 MB)
sentiment_analyzer = pipeline("sentiment-analysis")

print("✓ Sentiment analysis model loaded!")

In [ ]:
# Analyze a single sentence
result = sentiment_analyzer("I love this machine learning course!")
print("Input: 'I love this machine learning course!'")
print(f"Result: {result}")
print(f"\nSentiment: {result[0]['label']}")
print(f"Confidence: {result[0]['score']:.1%}")

In [ ]:
# Analyze multiple sentences at once
texts = [
    "This restaurant has amazing food!",
    "The movie was boring and too long.",
    "The weather is okay today.",
    "I'm really excited about the new update!",
    "Customer service was terrible and unhelpful.",
    "The product works as expected."
]

results = sentiment_analyzer(texts)

# Display results in a nice table
print(f"{'Text':<50} {'Sentiment':>10} {'Confidence':>12}")
print("=" * 75)
for text, result in zip(texts, results):
    label = result['label']
    score = result['score']
    print(f"{text:<50} {label:>10} {score:>11.1%}")

### Visualize Sentiment Results

In [ ]:
# Create a visualization of sentiment scores
labels = [r['label'] for r in results]
scores = [r['score'] if r['label'] == 'POSITIVE' else -r['score'] for r in results]
colors = ['green' if s > 0 else 'red' for s in scores]

# Short labels for display
short_texts = [t[:35] + '...' if len(t) > 35 else t for t in texts]

plt.figure(figsize=(12, 6))
plt.barh(range(len(texts)), scores, color=colors, alpha=0.7, edgecolor='black')
plt.yticks(range(len(texts)), short_texts, fontsize=10)
plt.xlabel('Sentiment Score (negative ← → positive)', fontsize=12)
plt.title('Sentiment Analysis Results', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.xlim([-1.1, 1.1])
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

**That's it!** Three lines of code to run a state-of-the-art sentiment analysis model.

Behind the scenes, HuggingFace:
1. Downloaded a model trained on millions of text examples
2. Tokenized your text (converted words to numbers)
3. Ran inference through the neural network
4. Returned human-readable labels and confidence scores

---

## Part 2: Zero-Shot Classification

**Zero-shot classification** can categorize text into categories it was never specifically trained on!

**How?** The model understands the *meaning* of both the text and the category labels, then matches them.

**This is remarkable:** You define your own categories, and the model figures it out.

In [ ]:
# Create zero-shot classification pipeline
classifier = pipeline("zero-shot-classification")

print("✓ Zero-shot classification model loaded!")

In [ ]:
# Classify a news headline into custom categories
text = "Apple announces new MacBook Pro with M4 chip"
candidate_labels = ["technology", "sports", "politics", "finance", "entertainment"]

result = classifier(text, candidate_labels)

print(f"Text: '{text}'\n")
print(f"{'Category':<15} {'Confidence':>10}")
print("-" * 27)
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 30)
    print(f"{label:<15} {score:>9.1%} {bar}")

In [ ]:
# Try different texts and categories
examples = [
    {
        "text": "The central bank raised interest rates by 0.25%",
        "labels": ["finance", "technology", "health", "sports"]
    },
    {
        "text": "Scientists discover high levels of microplastics in the Arctic",
        "labels": ["environment", "cooking", "fashion", "sports"]
    },
    {
        "text": "I need to book a flight and reserve a hotel for next week",
        "labels": ["travel", "food", "education", "healthcare"]
    }
]

for ex in examples:
    result = classifier(ex["text"], ex["labels"])
    top_label = result['labels'][0]
    top_score = result['scores'][0]
    print(f"'{ex['text'][:60]}...'")
    print(f"  → {top_label} ({top_score:.1%})\n")

### Why This is Powerful

**Traditional ML approach:**
1. Collect thousands of labeled examples per category
2. Train a model for weeks
3. Model only knows those specific categories
4. Need to retrain for new categories

**Zero-shot approach:**
1. Just describe your categories in words
2. The model figures out the rest
3. Works with **any** categories you can name!
4. No training data needed

---

## Part 3: Text Summarization

**Summarization** condenses long text into a shorter version while keeping the key information.

In [ ]:
# Create summarization pipeline
summarizer = pipeline("summarization")

print("✓ Summarization model loaded!")

In [ ]:
# Summarize a long text
long_text = """
Machine learning is a branch of artificial intelligence that focuses on building 
systems that learn from data. Unlike traditional programming where developers write 
explicit rules, machine learning algorithms automatically find patterns in data and 
use them to make predictions or decisions. There are three main types of machine 
learning: supervised learning, where the model learns from labeled examples; 
unsupervised learning, where the model finds patterns in unlabeled data; and 
reinforcement learning, where an agent learns by interacting with an environment 
and receiving rewards or penalties. Machine learning is used in many applications 
today, including spam detection, recommendation systems, image recognition, natural 
language processing, and autonomous vehicles. The field has seen rapid advancement 
in recent years, particularly with the development of deep learning techniques 
that use neural networks with many layers to learn complex patterns in data.
"""

summary = summarizer(long_text, max_length=60, min_length=20, do_sample=False)

print(f"Original length: {len(long_text.split())} words")
print(f"Summary length: {len(summary[0]['summary_text'].split())} words")
print(f"\nSummary:")
print(f"  {summary[0]['summary_text']}")

---

## Part 4: Loading Datasets from HuggingFace

HuggingFace also hosts thousands of datasets, ready to use in one line.

### The `datasets` Library

**Benefits over manual data loading:**
- One-line download and caching
- Standard format across all datasets
- Built-in train/test splits
- Efficient memory handling (streaming for large datasets)

In [ ]:
from datasets import load_dataset

# Load the IMDB movie review dataset
print("Loading IMDB dataset... (first download may take a moment)")
imdb = load_dataset("imdb")

print(f"\n✓ Dataset loaded!")
print(f"\nDataset structure:")
print(imdb)
print(f"\nTraining examples: {len(imdb['train']):,}")
print(f"Test examples: {len(imdb['test']):,}")

In [ ]:
# Explore the dataset
print("Features:", imdb['train'].features)

# Look at one example
example = imdb['train'][0]
print(f"\nSample review (first 300 chars):")
print(f"  {example['text'][:300]}...")
print(f"\nLabel: {example['label']} ({'positive' if example['label'] == 1 else 'negative'})")

In [ ]:
# Convert to Pandas for easy exploration
imdb_df = imdb['train'].to_pandas()

print("Dataset as Pandas DataFrame:")
print(imdb_df.head())

# Label distribution
print(f"\nLabel distribution:")
print(imdb_df['label'].value_counts().rename({0: 'negative', 1: 'positive'}))

# Review length statistics
imdb_df['word_count'] = imdb_df['text'].str.split().str.len()
print(f"\nReview length statistics:")
print(f"  Min: {imdb_df['word_count'].min()} words")
print(f"  Max: {imdb_df['word_count'].max()} words")
print(f"  Mean: {imdb_df['word_count'].mean():.0f} words")

### Using the Sentiment Model on IMDB Reviews

In [ ]:
# Apply sentiment analysis to a few IMDB reviews
sample_reviews = imdb['test'][:5]

# Truncate long reviews (model has a max input length)
short_reviews = [text[:512] for text in sample_reviews['text']]
true_labels = sample_reviews['label']

predictions = sentiment_analyzer(short_reviews)

print(f"{'True':<10} {'Predicted':<12} {'Confidence':>10}   Review Preview")
print("=" * 80)
for true_label, pred, review in zip(true_labels, predictions, short_reviews):
    true_str = 'positive' if true_label == 1 else 'negative'
    pred_str = pred['label'].lower()
    match = '✓' if true_str == pred_str else '✗'
    preview = review[:40].replace('\n', ' ') + '...'
    print(f"{true_str:<10} {pred_str:<12} {pred['score']:>9.1%}  {match} {preview}")

---

## Part 5: Available Pipelines Overview

HuggingFace provides pipelines for many tasks:

| Task | Pipeline Name | Example |
|------|--------------|----------|
| **Sentiment Analysis** | `"sentiment-analysis"` | Is this review positive or negative? |
| **Zero-Shot Classification** | `"zero-shot-classification"` | Categorize text into custom labels |
| **Text Generation** | `"text-generation"` | Complete a sentence or write text |
| **Summarization** | `"summarization"` | Shorten a long text |
| **Translation** | `"translation"` | Translate between languages |
| **Question Answering** | `"question-answering"` | Answer questions about a text |
| **Fill Mask** | `"fill-mask"` | Predict missing words |
| **Named Entity Recognition** | `"ner"` | Find names, places, organizations |
| **Image Classification** | `"image-classification"` | Classify images |
| **Object Detection** | `"object-detection"` | Find objects in images |

All work the same way: `pipeline("task-name")(your_input)`

---

## Part 6: Exercises

### Exercise 1: Customer Review Analyzer

**Scenario:** You work for an e-commerce company and need to analyze customer feedback.

**Task:**
1. Create a list of 5-10 product reviews (write your own or use the examples below)
2. Use the sentiment analysis pipeline to classify each review
3. Visualize the results (bar chart of positive vs. negative)
4. Calculate the overall sentiment score

In [ ]:
# Sample reviews (you can replace with your own!)
reviews = [
    "The product arrived quickly and works perfectly. Very happy with my purchase!",
    "Terrible quality. Broke after just two days of use.",
    "Good value for the price. Not amazing but does the job.",
    "Absolutely love it! Best purchase I've made this year.",
    "The color is different from what was shown. Disappointed.",
    "Shipping was slow but the product itself is great.",
    "Would not recommend. Wasted my money.",
    "Exceeded my expectations! Will buy again."
]

# TODO: Analyze all reviews with the sentiment pipeline
review_results = None  # Use sentiment_analyzer(reviews)

# TODO: Print results in a formatted table

# TODO: Calculate overall stats
# How many positive? How many negative? Average confidence?

In [ ]:
# TODO: Visualize the results
# Create a bar chart showing sentiment score for each review
# Positive scores should be green bars, negative should be red

plt.figure(figsize=(12, 6))
# Your visualization code here
plt.show()

---

### Exercise 2: News Article Classifier

**Scenario:** You're building a news aggregator that automatically categorizes articles.

**Task:**
1. Write 5 news-style headlines
2. Define your own set of categories
3. Use zero-shot classification to categorize each headline
4. Display the top prediction and confidence for each

In [ ]:
# TODO: Write your news headlines
headlines = [
    # Add your headlines here
]

# TODO: Define your categories
categories = None  # e.g., ["politics", "sports", "technology", "health", "business"]

# TODO: Classify each headline
# Use classifier(headline, categories) for each

# TODO: Display results in a formatted table

---

### Exercise 3: Explore a HuggingFace Dataset

**Task:**
1. Load a dataset from HuggingFace (suggestions below)
2. Explore its structure (features, size, splits)
3. Convert to Pandas and show basic statistics
4. Apply a relevant pipeline to a few examples

**Suggested datasets:**
- `"rotten_tomatoes"` - Movie review sentiment
- `"ag_news"` - News article classification
- `"squad"` - Question answering
- `"emotion"` - Emotion detection in text

In [ ]:
# TODO: Load a dataset
my_dataset = None  # Use load_dataset("dataset_name")

# TODO: Explore the dataset
# - Print the structure
# - Show how many examples
# - Display first few examples

# TODO: Convert to Pandas and show statistics

# TODO: Apply a pipeline to some examples

---

## When to Use Pre-Trained Models vs. Training From Scratch

| Scenario | Approach | Why |
|----------|----------|-----|
| **Standard NLP tasks** | Pre-trained model | Already excellent, no training needed |
| **Custom categories** | Zero-shot or fine-tuning | Zero-shot for quick experiments, fine-tune for production |
| **Specialized domain** (medical, legal) | Fine-tune pre-trained | Start from general knowledge, adapt to domain |
| **Very simple task** (spam filter) | Train from scratch | Small dataset, simple patterns |
| **Novel architecture research** | Train from scratch | Testing new ideas |

**General rule:** Start with pre-trained models. Only train from scratch if they don't work for your task.

---

## Summary

Congratulations! You've learned:

✓ What pre-trained models are and why they matter  
✓ How to use HuggingFace pipelines for NLP tasks  
✓ Sentiment analysis with one line of code  
✓ Zero-shot classification with custom categories  
✓ Text summarization  
✓ Loading and exploring HuggingFace datasets  

### Key Takeaways:

1. **Pre-trained models save enormous time and resources**
2. **HuggingFace makes state-of-the-art ML accessible** - No PhD required!
3. **Pipelines are the easiest way** to use pre-trained models
4. **Zero-shot classification is magic** - Custom categories without training
5. **Always start with pre-trained models** - Train from scratch only if needed

### What's Next?

- **Fine-tuning:** Adapt a pre-trained model to your specific data
- **Model Hub:** Browse [huggingface.co/models](https://huggingface.co/models) for specialized models
- **Gradio:** Build interactive demos with `gradio` library
- **Day 5 Project:** Use HuggingFace in your own project!

### Additional Resources:

- [HuggingFace Documentation](https://huggingface.co/docs)
- [HuggingFace Course (free)](https://huggingface.co/course)
- [Model Hub](https://huggingface.co/models)
- [Dataset Hub](https://huggingface.co/datasets)

---